# Reproducible Statistical Q-Factor Estimation for a 25-Gb/s PAM4 IM–DD Optical Link

This notebook evaluates nine OptiSystem 23.1 transmitter/receiver waveform pairs under two independent parameter sweeps: single-mode fiber length (0–20 km) and receiver Bessel low-pass cutoff (6.5–8.5 GHz).

**Reported metric:** reference-labelled, minimum adjacent-level statistical Q, obtained from conditional received-amplitude means and standard deviations. This is an *offline waveform-derived separation metric*, not measured BER or OptiSystem's BER analyzer Q. The analysis does not involve a trained machine-learning model.

## 1. Data and computational environment

Data, verification checksums and analysis source are maintained in the associated [GitHub repository](https://github.com/SHAHRIAR-HOSSAIN-RIMON/OptiSystem_lab_final). The repository is cloned automatically on first execution in Colab; subsequent runs reuse the existing checkout. Local Jupyter use is supported from the project directory.

**Input:** one TX waveform and one RX waveform for each of nine experimental conditions, each containing 32,768 time-amplitude samples.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/SHAHRIAR-HOSSAIN-RIMON/OptiSystem_lab_final.git"
CHECKOUT_DIR = Path(os.environ.get("PAM4_CHECKOUT_DIR", "/content/OptiSystem_lab_final"))


def find_project_directory(location: Path):
    """Return the directory containing data/manifest.csv, if present."""
    for candidate in (location, location / "Optisystem"):
        if (candidate / "data" / "manifest.csv").is_file():
            return candidate.resolve()
    return None


# Local Jupyter and Colab use the same repository structure.
ROOT = next(
    (found for folder in (Path.cwd(), *Path.cwd().parents, CHECKOUT_DIR)
     if (found := find_project_directory(folder)) is not None),
    None,
)

if ROOT is None:
    if not CHECKOUT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet", REPOSITORY_URL, str(CHECKOUT_DIR)],
            check=True,
        )
    ROOT = find_project_directory(CHECKOUT_DIR)

if ROOT is None:
    raise FileNotFoundError(
        "data/manifest.csv was not found. If the existing checkout is incomplete, "
        "remove it or restart the Colab runtime and rerun this cell. "
        f"Checkout path: {CHECKOUT_DIR}"
    )

print("Analysis project:", ROOT)
print("Manifest:", ROOT / "data" / "manifest.csv")

## 2. Verification and numerical reconstruction

The verification program checks the SHA-256 digest of every input waveform against `data/manifest.csv`, estimates the three adjacent-level Q values for each condition, and compares their minimum to the archived result (absolute tolerance \(2\times10^{-5}\)). It also regenerates summary tables and figures.

In [ ]:
command = [
    sys.executable,
    str(ROOT / "scripts" / "reproduce_all.py"),
    "--root", str(ROOT),
]
result = subprocess.run(
    command, cwd=ROOT, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(
        "Waveform verification or Q-factor reconstruction failed "
        f"(exit code {result.returncode}). See the printed diagnostics above."
    )

## 3. Parameter-sweep results

The reported minimum statistical Q is computed using all 496 retained symbols. Held-out Q is calculated from the second half of the retained symbols using a timing offset calibrated on the first half.

In [ ]:
import csv
from IPython.display import Markdown, Image, display


def show_sweep(csv_path: Path, parameter: str, title: str, figure_path: Path):
    with csv_path.open(newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    lines = [
        f"### {title}",
        f"| {parameter} | Full-record min Q | Held-out min Q |",
        "|---:|---:|---:|",
    ]
    for row in rows:
        lines.append(
            f"| {row[parameter]} | {float(row['q_min']):.6f} | "
            f"{float(row['q_min_holdout']):.6f} |"
        )
    display(Markdown("\n".join(lines)))
    display(Image(filename=str(figure_path)))


show_sweep(
    ROOT / "results/summary/fiber_sweep.csv", "fiber_km",
    "Fiber-length sweep (7.5 GHz cutoff)",
    ROOT / "results/figures/fiber_sweep_Q.png",
)
show_sweep(
    ROOT / "results/summary/bandwidth_sweep.csv", "cutoff_GHz",
    "Receiver-bandwidth sweep (10 km SMF)",
    ROOT / "results/figures/bandwidth_sweep_Q.png",
)

## 4. Adjacent-level statistics

The received samples are grouped according to their corresponding known transmitted PAM4 symbol. For each level \(k\), let \(\mu_k\) and \(\sigma_k\) denote the conditional mean and sample standard deviation of the received amplitude. The adjacent-level Q estimates are

\[
Q_k = \frac{\mu_{k+1}-\mu_k}{\sigma_{k+1}+\sigma_k},\qquad k\in\{0,1,2\},
\quad Q_{\min}=\min_k Q_k.
\]

The 10-km, 7.5-GHz condition below illustrates the estimated level populations and their adjacent-level separations.

In [ ]:
level_csv = ROOT / "results/per_run/fiber_10km_7p5GHz_levels.csv"
with level_csv.open(newline="", encoding="utf-8") as file:
    level_rows = list(csv.DictReader(file))

lines = [
    "| PAM4 level | Count | RX mean (native units) | RX std. dev. (native units) | Adjacent Q |",
    "|---:|---:|---:|---:|---:|",
]
for row in level_rows:
    q = row["q_to_next_level"] or "—"
    lines.append(
        f"| {row['PAM4_level']} | {row['n']} | "
        f"{float(row['mean_received_amplitude_input_units']):.8g} | "
        f"{float(row['std_received_amplitude_input_units']):.8g} | {q} |"
    )
display(Markdown("\n".join(lines)))

## 5. Assumptions and limitations

- The PAM4 symbol duration is 80 ps, sampled at 64 waveform points per symbol. Eight symbols at each record boundary are excluded.
- The receiver sampling offset is selected from 128 candidate offsets by maximizing the first-half minimum adjacent-level Q. The same offset is applied to the complete record and to the held-out half. Full-record results are partly in-sample; held-out results are diagnostics, not an independent experimental replicate.
- Each operating condition is represented by a single, 1024-bit simulation record. Several conditions use different pseudorandom transmitted patterns. No uncertainty intervals or replicated-run statistics are reported.
- The observed level spreads combine modeled noise and deterministic waveform distortion, including intersymbol interference. Q-factor estimates therefore cannot be converted into validated BER without additional assumptions and evaluation.
- The recorded waveform data and analysis program are reproducible; the editable OptiSystem schematic itself is not included in this repository.

**Data products:** `results/summary/*.csv`, `results/per_run/*.csv`, and `results/figures/*.png`.